In [ ]:
! pip install datasets transformers

In [1]:
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from pathlib import Path
import json
import os
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
# Configuration
DATASET_PATH = Path('/content/drive/MyDrive/hf_datasets/hungarian')
LABEL_MAPPING_PATH = Path('/content/drive/MyDrive/hf_datasets/label_mapping.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/trained_models/hubert')

In [4]:
# Model choice
MODEL_NAME = "akdeniz27/bert-base-hungarian-cased-ner" # Hungarian

In [5]:
# Training hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 2
MAX_LENGTH = 512

In [6]:
# Load label mapping
print("Loading label mapping...")
with open(LABEL_MAPPING_PATH, 'r') as f:
    label_info = json.load(f)

label2id = {k: int(v) for k, v in label_info["label2id"].items()}
id2label = {int(k): v for k, v in label_info["id2label"].items()}
label_list = label_info["label_list"]

print(f"Labels: {label_list}")
print(f"Number of labels: {len(label_list)}")

Loading label mapping...
Labels: ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-DATE', 'I-DATE']
Number of labels: 9


In [7]:
# Load dataset
print(f"\nLoading dataset from {DATASET_PATH}...")

if not DATASET_PATH.exists():
    print(f"Error: Dataset directory not found at {DATASET_PATH}.")
    print("Please ensure your Google Drive is mounted and the dataset exists at this path.")
else:
    dataset = load_from_disk(DATASET_PATH)

    print(f"Train examples: {len(dataset['train'])}")
    print(f"Val examples: {len(dataset['test'])}")


Loading dataset from /content/drive/MyDrive/hf_datasets/hungarian...
Train examples: 400
Val examples: 100


In [8]:
# Load tokenizer
print(f"\nLoading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization function
def tokenize_and_align_labels(examples):
    """
    Tokenize inputs and align NER labels with subword tokens
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens get label -100
            if word_idx is None:
                label_ids.append(-100)
            # First token of a word gets the label
            elif word_idx != previous_word_idx:
                if word_idx < len(label):
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            # Subsequent tokens of the same word get -100
            else:
                label_ids.append(-100)

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize dataset
print("\nTokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(f"Tokenized train examples: {len(tokenized_dataset['train'])}")
print(f"Tokenized val examples: {len(tokenized_dataset['test'])}")

# Load model
print(f"\nLoading model: {MODEL_NAME}...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


Loading tokenizer: akdeniz27/bert-base-hungarian-cased-ner...


config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/272k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/519k [00:00<?, ?B/s]

...

[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `7`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: akdeniz27/bert-base-hungarian-cased-ner
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([9])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7, 768]) vs model:torch.Size([9, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [9]:
# Metrics

def extract_spans_from_bio(tags):
    """Converts BIO sequence to entity spans (start_idx, end_idx, entity_type)."""
    spans = []
    current_start = None
    current_type = None

    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if current_type is not None:
                spans.append((current_start, i, current_type))
            current_start = i
            current_type = tag[2:]
        elif tag.startswith("I-"):
            tag_type = tag[2:]
            if current_type is None:
                current_start = i
                current_type = tag_type
            elif tag_type != current_type:
                spans.append((current_start, i, current_type))
                current_start = i
                current_type = tag_type
        else:  # 'O'
            if current_type is not None:
                spans.append((current_start, i, current_type))
                current_start = None
                current_type = None

    if current_type is not None:
        spans.append((current_start, len(tags), current_type))

    return spans


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Reconstruct clean sequence-level BIO tags per sentence (skip special -100 tokens)
    true_predictions = [
        [id2label[p_idx] for (p_idx, l_idx) in zip(pred, lab) if l_idx != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l_idx] for (p_idx, l_idx) in zip(pred, lab) if l_idx != -100]
        for pred, lab in zip(predictions, labels)
    ]

    total_tp = 0
    total_fp = 0
    total_fn = 0

    # Exact strict chunk matching across all sentences
    for true_tags, pred_tags in zip(true_labels, true_predictions):
        gold_spans = set(extract_spans_from_bio(true_tags))
        pred_spans = set(extract_spans_from_bio(pred_tags))

        tp = len(gold_spans & pred_spans)
        fp = len(pred_spans - gold_spans)
        fn = len(gold_spans - pred_spans)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = (
        total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    )
    recall = (
        total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [10]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    logging_steps=10,
    report_to="none",
    optim="adamw_torch",
    save_total_limit=2,
    dataloader_num_workers=0,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # Now using your custom metrics
)

# Train
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH.name}")
print(f"Output: {OUTPUT_DIR}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print("=" * 60 + "\n")

trainer.train()


STARTING TRAINING
Model: akdeniz27/bert-base-hungarian-cased-ner
Dataset: hungarian
Output: /content/drive/MyDrive/trained_models/hubert
Epochs: 10
Batch size: 8
Learning rate: 2e-05



Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.472677,0.159835,0.641624,0.436464,0.519523
2,0.272116,0.110257,0.703883,0.600829,0.648286
3,0.216579,0.099140,0.713575,0.693370,0.703327
4,0.182327,0.095571,0.695973,0.716160,0.705922
5,0.171746,0.091848,0.708024,0.725138,0.716479
6,0.152906,0.088696,0.708869,0.723066,0.715897
7,0.137175,0.085632,0.717391,0.752072,0.734322
8,0.112445,0.086360,0.705587,0.776243,0.739231
9,0.118972,0.084358,0.708517,0.775552,0.740521
10,0.111744,0.084733,0.711477,0.774862,0.741818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.23813895606994628, metrics={'train_runtime': 557.3019, 'train_samples_per_second': 7.177, 'train_steps_per_second': 0.449, 'total_flos': 1045253173248000.0, 'train_loss': 0.23813895606994628, 'epoch': 10.0})

In [11]:
# Evaluate
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = trainer.evaluate()

print(f"\nValidation Results:")
print(f"  Precision: {results.get('eval_precision', 0):.4f}")
print(f"  Recall:    {results.get('eval_recall', 0):.4f}")
print(f"  F1 Score:  {results.get('eval_f1', 0):.4f}")
print(f"  Accuracy:  {results.get('eval_accuracy', 0):.4f}")


FINAL EVALUATION


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.111744,0.084733,10,0.711477,0.774862,0.741818



Validation Results:
  Precision: 0.7115
  Recall:    0.7749
  F1 Score:  0.7418
  Accuracy:  0.0000


In [ ]:
# Save model
print(f"\nSaving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save label mapping with the model
with open(OUTPUT_DIR / "label_mapping.json", 'w') as f:
    json.dump({
        "label_list": label_list,
        "label2id": label2id,
        "id2label": id2label
    }, f, indent=2)

# Save results
with open(OUTPUT_DIR / "eval_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print("\nTraining complete!")
print(f"Model saved to: {OUTPUT_DIR}")


Saving model to /content/drive/MyDrive/trained_models/hubert...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
Model saved to: /content/drive/MyDrive/trained_models/hubert
